## Training Strategy

We will use a **Multinomial Naive Bayes classifier** with a **Bag-of-Words representation** to classify emails as either spam or ham.

### How Naive Bayes Works

Naive Bayes calculates the **prior probability** of each class and the **likelihood** of observing the words in a document given that class. These values are then combined to estimate the posterior probability of each class.

The classifier returns the argmax between each posterior probability.

### Equations

#### 1. Prior Probability

The prior probability represents how frequently a class occurs within the training corpus.

$$
P(c) = \frac{D_c}{D}
$$

Where:

* ${D_c}$ is the number of documents belonging to class $c$.
* ${D}$ is the total number of documents in the corpus.

#### 2. Word Likelihood

The likelihood represents the probability of observing a particular word given a class.

$$
P(w_i \mid c) =
\frac{N_{w_i,c} + \alpha}
{N_c + \alpha V}
$$

Where:

* $N_{w_i,c}$ is the number of times word $w_i$ appears in class $c$.
* $N_c$ is the total number of word occurrences in class $c$, including repetitions.
* $V$ is the total vocabulary size, excluding repetitions.
* $\alpha$ is the smoothing parameter used to prevent zero likelihoods.

Laplace smoothing ensures that a word that does not occur in a particular class does not cause the entire posterior probability to become zero.

#### 3. Posterior Probability

Given a document represented by its words $X_1, X_2, \ldots, X_n$, Naive Bayes estimates the probability of each class using:

$$
P(c \mid X) \propto
P(c)\prod_{i=1}^{n}P(X_i \mid c)
$$

The denominator from Bayes' theorem is omitted because it is identical for every candidate class and therefore does not affect which class has the highest probability.

The predicted class is therefore:

$$
\hat{c} =
\underset{c}{\operatorname{argmax}}
\left[
P(c)\prod_{i=1}^{n}P(X_i \mid c)
\right]
$$

### Arithmetic Underflow

Directly multiplying the likelihoods can result in extremely small numbers. As the number of words in a document increases, the product of many probabilities between 0 and 1 can become smaller than the range representable by standard floating-point numbers.

This results in **arithmetic underflow**, where the calculated probability underflows to (0.0).

To avoid this, we transform the calculation from probability space into **log space**.

Using the logarithmic identity:

$$
\log(A \times B) = \log(A) + \log(B)
$$

we can transform the posterior calculation:

$$
\log\left(
P(c)\prod_{i=1}^{n}P(X_i \mid c)
\right)
=

\log P(c)
+
\sum_{i=1}^{n}\log P(X_i \mid c)
$$

Therefore, the log-space scoring function becomes:

$$
\log P(c \mid X)
\propto
\log P(c)
+
\sum_{i=1}^{n}\log P(X_i \mid c)
$$

The class with the highest log-probability is selected as the prediction.

In [229]:
# Imports 
import json
import re
from collections import Counter

import numpy as np
import pandas as pd

In [230]:
# Load data and drop null values 
df = pd.read_csv("../data/raw/training_data.csv")
df = df.dropna(subset=["text"])

## Cleaning strategy
The same cleaning strategy from the EDA notebook is used as a baseline 
(v1). four preprocessing configurations are trained and evaluated here, ranging from full 
filtering (v1) down to minimal filtering (v4), to observe the impact 
of each filtering step on precision, recall, and F1 score.

| Version | Stopwords removed? | NOISE_WORDS removed? | escapenumber/escapelong stripped? | Punctuation stripped? |
|---------|--------------------|-----------------------|------------------------------------|------------------------|
| v1      | Yes                | Yes                   | Yes                                 | Yes                    |
| v2      | Yes                | No                     | Yes                                 | Yes                    |
| v3      | No                 | No                     | Yes                                 | Yes                    |
| v4      | No                 | No                     | No                                  | Yes                    |

The effects will be observed during evaluation.

In [231]:
# clean and tokanize
pattern = re.compile(r'[^a-zA-Z\s]+')
noise_pattern = re.compile(r'escapenumber|escapelong', re.IGNORECASE)
stopwords = {'y', 'did', 'i', 'again', "hadn't", 'my', 'over', 'too', 'here', "that'll", 'couldn', 'than', "they've", 'same', "she's", 'they', 'doing', 'if', 'down', "don't", 'out', 'no', 'her', 're', 'such', 't', "i've", 'only', 'was', 'shouldn', "they'd", 'won', 'more', 'needn', 'do', "shouldn't", "wasn't", "aren't", 'aren', 'how', 'shan', 'doesn', 'few', 'll', 'myself', "couldn't", "he'd", 'other', 'wasn', 'his', 'on', 'these', 'both', "didn't", 'you', "you've", 'their', 'what', "you'd", 'being', 'by', 'been', 's', 'ain', 'why', 'where', 'until', 'as', 'off', 'after', 'is', 'be', 'below', 'hasn', 'yours', 'we', 'has', 'own', 've', 'haven', 'whom', 'wouldn', 'hers', "we'd", 'between', 'o', "he's", 'have', 'herself', 'does', 'now', "shan't", 'don', 'in', 'so', 'from', 'a', 'under', 'further', 'those', 'me', 'most', "weren't", 'against', 'its', 'she', 'at', "you'll", 'yourself', "i'll", 'm', 'once', 'mustn', 'while', 'should', 'ours', 'didn', 'then', 'when', 'that', 'were', "it'd", 'which', 'above', 'all', "doesn't", "mustn't", "she'd", "it's", 'before', 'of', 'and', 'weren', "she'll", 'our', 'will', 'isn', "i'm", 'had', 'to', 'the', 'through', 'with', 'there', 'during', 'or', "haven't", 'himself', 'it', 'theirs', "wouldn't", 'each', 'not', 'just', 'this', "we'll", "he'll", "needn't", "we've", 'ourselves', 'about', "isn't", 'your', 'nor', 'because', 'can', 'he', 'am', "i'd", "should've", 'any', 'd', 'some', 'having', 'ma', 'itself', 'into', "we're", "hasn't", "they'll", 'who', 'are', 'but', 'themselves', "mightn't", "it'll", 'very', 'for', 'hadn', "you're", 'him', 'them', 'an', "they're", 'mightn', "won't", 'yourselves', 'up'}
NOISE_WORDS = {"subject", "pm", "org", "com", "http", "www"}

def clean(email: str) -> list[str]:
    email = noise_pattern.sub('', email)
    words = email.lower().split()
    words = [pattern.sub('', w) for w in words]
    words = [
        w for w in words
        if w and len(w) > 1 and w not in stopwords and w not in NOISE_WORDS
    ]
    return words

In [ ]:
# build vocabulary 
spam_column = df[df["label"] == "Spam"]
spam_text_column = spam_column["text"]

ham_column = df[df["label"] == "Ham"]
ham_text_column = ham_column["text"]


spam_vocab = []
ham_vocab = []

for ham_email in ham_text_column:
    ham_vocab.extend(clean(ham_email))

for spam_email in spam_text_column:
    spam_vocab.extend(clean(spam_email))

global_vocab = set(ham_vocab) | set(spam_vocab)

## Indexed Vocabulary

Here we build a dictionary containing every unique word in the global vocabulary and assign an index to each word.

This is important because during prediction, we need to look up the index of a specific word in order to find its corresponding spam and ham log-likelihoods in the log-likelihood matrix.

For example, if the word `Free` is assigned index (0) in the indexed vocabulary, then:

* Index (0) of the spam vector contains the log-likelihood of `Free` given spam.
* Index (0) of the ham vector contains the log-likelihood of `Free` given ham.

This ensures that the same vocabulary index always refers to the same word across the document vector, spam vector, and ham vector.


In [ ]:
V = len(global_vocab)
indexed_vocab = {word: i for i, word in enumerate(global_vocab)}
indexed_vocab

In [ ]:
# Compute priors 
corpus = len(df)
prior_spam = np.log(len(spam_column) / corpus) # log P(spam)
prior_ham = np.log(len(ham_column) / corpus) # log P(ham)

## Computing Likelihoods

Here we want to construct a $2 \times V$ matrix, where $V$ is the vocabulary size.

```python
[
    [-17.9595, -6.2345, ...],     # spam vector
    [-8.238492, -14.17874, ...]   # ham vector
]
```

Each position in both vectors corresponds to the exact same word in the indexed vocabulary.

For example, if `free` is assigned index (42) in the vocabulary, then position (42) in the spam vector contains the log-likelihood of `free` given spam, while position (42) in the ham vector contains the log-likelihood of `free` given ham.

This ensures that the document vector and both class vectors share the same representation, allowing us to calculate their dot products during prediction.


In [ ]:
# Compute likelihoods
alpha = 1
spam_likelihoods = np.zeros(len(global_vocab))
ham_likelihoods = np.zeros(len(global_vocab))

Ns = len(spam_vocab) # total number of words in spam
Nh = len(ham_vocab) # total number of words in ham

spam_word_count = Counter(spam_vocab)
ham_word_count = Counter(ham_vocab)

for word, index in indexed_vocab.items():
    Nws = spam_word_count.get(word, 0)
    Nwh = ham_word_count.get(word, 0)

    spam_likelihoods[index] = np.log((Nws + alpha) / (Ns + V * alpha))
    ham_likelihoods[index] = np.log((Nwh + alpha) / (Nh + V * alpha))

log_likelihoods = np.vstack([spam_likelihoods, ham_likelihoods])

In [ ]:
# save model to model.json
model = {
    "vocab": indexed_vocab,
    "log_priors": [float(prior_spam), float(prior_ham)],
    "log_likelihoods": log_likelihoods.tolist(),
}
version =1
path = f"../models/modelv{version}.json"
with open(path, "w") as f:
    json.dump(model, f, indent=2)

## Implementation Strategy

The classifier represents the vocabulary, class likelihoods, and documents using vectors of the same size (V).

During prediction, a cleaned document is converted into a Bag-of-Words vector by using the indexed vocabulary to map each word to its corresponding position and incrementing that position for every occurrence.

The resulting document vector is then multiplied with each class's log-likelihood vector using a dot product:

$$
X \cdot \log P(W \mid c)
=

\sum_{i=1}^{V} X_i \log P(w_i \mid c)
$$

The dot product is added to the class's log prior to produce the final class score:

$$
\text{score}(c)
=

\log P(c)
+
X \cdot \log P(W \mid c)
$$

This score is calculated for both spam and ham, and the class with the highest score is selected as the prediction.


In [ ]:
# Prediction
def load_model(path="../models/modelv4.json"):
    with open(path, "r") as f:
        model = json.load(f)
    vocab = model["vocab"]
    log_prior_s, log_prior_h = np.array(model["log_priors"])
    log_likelihoods = np.array(model["log_likelihoods"])
    return vocab, log_prior_s, log_prior_h, log_likelihoods

def predict(email, vocab, log_prior_s, log_prior_h, log_likelihoods):
    cleaned = clean(email)
    doc_vector = np.zeros(len(vocab))
    for word in cleaned:
        index = vocab.get(word)
        if index is None:
            continue
        doc_vector[index] += 1

    log_post_spam = log_prior_s + (doc_vector @ log_likelihoods[0])
    log_post_ham = log_prior_h + (doc_vector @ log_likelihoods[1])

    return "spam" if np.argmax([log_post_spam, log_post_ham]) == 0 else "ham"

In [ ]:
# sanity checks 
sample_df = pd.read_csv("../data/raw/testing_dataset.csv")
correct = 0
total = len(sample_df)

vocab, log_prior_s, log_prior_h, log_likelihoods = load_model()

In [ ]:
for index, row in sample_df.iterrows():
    label = row["label"]
    email = row["text"]

    if label.lower() == predict(email, vocab, log_prior_s, log_prior_h, log_likelihoods):
        correct += 1

print(f"Current Model accuracy: {(correct / total) * 100}")
print(f"Baseline : {96.6478433257239}")


Current Model accuracy: 95.91893948496136
Baseline : 96.6478433257239
